# Traffic Object Detection

This notebook demonstrates a pretrained YOLOv5 pipeline for vehicle and pedestrian detection using BDD100K-style image inputs.

In [1]:
import os
import glob
import matplotlib.pyplot as plt
os.makedirs('../data/bdd100k', exist_ok=True)

images = glob.glob('../data/bdd100k/*.jpg') + glob.glob('../data/bdd100k/*.png')
print(f'Found {len(images)} sample images')

Found 0 sample images


In [2]:
try:
    import torch
    model_yolo = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True, verbose=False)
    model_yolo.eval()
    VEHICLE_CLASSES = ['car', 'bus', 'truck', 'motorcycle', 'bicycle']
    PERSON_CLASSES = ['person']
    print('YOLOv5 loaded successfully.')

    def detect_objects(image_path):
        results = model_yolo(image_path)
        df_det = results.pandas().xyxy[0]
        vehicles = df_det[df_det['name'].isin(VEHICLE_CLASSES)]
        pedestrians = df_det[df_det['name'].isin(PERSON_CLASSES)]
        return {
            'vehicle_count': len(vehicles),
            'pedestrian_count': len(pedestrians),
            'detections': df_det[['name', 'confidence']].to_dict('records')
        }

    if images:
        for img_path in images[:2]:
            result = detect_objects(img_path)
            print(os.path.basename(img_path), result)
            res = model_yolo(img_path)
            res.render()
            plt.figure(figsize=(10, 6))
            plt.imshow(res.ims[0])
            plt.axis('off')
            plt.show()
    else:
        print('No sample images found. Add images to ../data/bdd100k/')
except Exception as e:
    print('YOLOv5 not available:', e)
    print('Install with: pip install torch torchvision')

YOLOv5 not available: No module named 'ultralytics'
Install with: pip install torch torchvision
